# GameTheory-20 : Capstone — la sous-série Coalitions en une heure

**Navigation** : [<< 17-MultiAgent-RL](GameTheory-17-MultiAgent-RL-Python.ipynb) | [Index](README.md) | [Capstone 21 >>](GameTheory-21-Capstone-Choix-Social-Python.ipynb) · sous-série [Coalitions](Coalitions/README.md)

**Side tracks** : ce carnet EST l'escalier vers la sous-série [Coalitions](Coalitions/README.md) — il la présente et n'en remplace aucun carnet.

Ce carnet est le **capstone** de la sous-série Coalitions, consacrée à une question que la théorie des jeux coopératifs pose depuis von Neumann et Morgenstern : **combien vaut une coalition, et à quel niveau de garantie**. La sous-série y répond par paliers — un indice calculé, une borne no-regret, une institution réelle, un optimum dual, une preuve formelle — et ce capstone les fait dialoguer sur les mêmes objets : chaque expérience confronte **deux niveaux de garantie** sur **un seul exemple**.

Il suppose lu le parcours principal de la série (théorie des jeux non-coopérative, équilibres) et referme la branche coopération ouverte par les carnets [01](Coalitions/Coalitions-01-Pouvoir-Coalitionnel-SMT-Python.ipynb) à [05](Coalitions/Coalitions-05-Lean-Affectation-Lean.ipynb).

## Le fil en une phrase

Poids n'est pas pouvoir ; une coalition ne vaut pas son poids mais ce que ses membres pivots peuvent obtenir ; et la valeur d'une coalition se décline en **garanties de force croissante** : mesurée (simulation), prouvée (certificat vérifiable), ou démontrée (théorème formel).

## 1. Ce que la sous-série démontre

| Carnet | Ce qu'il démontre | Niveau de garantie |
|---|---|---|
| [Coalitions-01 — Pouvoir coalitionnel par SMT](Coalitions/Coalitions-01-Pouvoir-Coalitionnel-SMT-Python.ipynb) | poids ≠ pouvoir : trois indices contrefactuels (Shapley-Shubik, Banzhaf, Deegan-Packel), puis énumération SMT en seconde voie | indice **calculé** (énumération exacte) |
| [Coalitions-02 — Assistance Games](Coalitions/Coalitions-02-Assistance-Games-Python.ipynb) | coopération humain-agent : oracle en hindsight contre apprenant no-regret EXP3, bench √T contre T^(3/4) | **borne** no-regret (garantie asymptotique) |
| [Coalitions-03 — Échange de reins](Coalitions/Coalitions-03-Echange-de-Reins-Lean-Python.ipynb) | le matching comme institution : cycles du graphe de compatibilité, chaînes de donneurs altruistes, arbitrage cardinalité/équité | institution **réelle** (données et contraintes du monde) |
| [Coalitions-04 — Munkres et affectation](Coalitions/Coalitions-04-Munkres-Affectation-Python.ipynb) | l'affectation optimale : Kuhn-Munkres from scratch, lecture par dualité, jeu d'affectation de Shapley-Shubik, deux notions d'optimum | optimum **dual** (certificat vérifiable) |
| [Coalitions-05 — Lean Affectation](Coalitions/Coalitions-05-Lean-Affectation-Lean.ipynb) | le lake `assignment_lean` par son certificat : quatre modules, valeur d'une affectation, ce que la preuve garantit | preuve **formelle** (théorème machine-vérifié) |

Le fil que la sous-série tisse : une même question — combien vaut coopérer — reçoit des réponses de force croissante. Le capstone fait tenir trois de ces niveaux dans la même fenêtre d'expérience.

## 2. Ce que ce notebook suppose

- **Un jeu coopératif pondéré** : des joueurs dotés de poids, un quota, des coalitions gagnantes — posé par [Coalitions-01](Coalitions/Coalitions-01-Pouvoir-Coalitionnel-SMT-Python.ipynb), dont nous reprenons le noyau tel quel.
- **Un graphe de compatibilité dirigé** entre paires donneur-receveur — posé par [Coalitions-03](Coalitions/Coalitions-03-Echange-de-Reins-Lean-Python.ipynb).
- **Une matrice de coûts et sa dualité** : les potentiels duaux `u_i + v_j <= c_ij` — posés par [Coalitions-04](Coalitions/Coalitions-04-Munkres-Affectation-Python.ipynb), dont nous reprenons l'algorithme de Kuhn-Munkres intégralement.
- Les sorties du lake `assignment_lean` sont **lues** (noms des théorèmes, structure des modules), jamais ré-exécutées ici : la preuve de build vit dans [Coalitions-05](Coalitions/Coalitions-05-Lean-Affectation-Lean.ipynb) et sa CI.

Aucun prérequis hors la série : numpy, pandas, networkx et itertools suffisent.

In [1]:
from dataclasses import dataclass
from itertools import combinations
from math import factorial
from pathlib import Path

import networkx as nx
import numpy as np
import pandas as pd


def find_repo_root():
    """Remontee d'ancetres jusqu'au lac assignment_lean (pattern Coalitions-03,
    independant du CWD de la session)."""
    p = Path.cwd().resolve()
    for candidat in [p, *p.parents]:
        if (candidat / "MyIA.AI.Notebooks/GameTheory/assignment_lean").is_dir():
            return candidat
    raise FileNotFoundError("racine du depot introuvable depuis " + str(p))


REPO_ROOT = find_repo_root()
print("imports OK — numpy", np.__version__, "| pandas", pd.__version__, "| networkx", nx.__version__)
print("racine resolue :", REPO_ROOT.name)

imports OK — numpy 2.4.2 | pandas 2.3.3 | networkx 3.6.1
racine resolue : CoursIA-gt20-capstone


## 3. Expérience 1 — Le président pénalisé

*Ancrée dans [Coalitions-01](Coalitions/Coalitions-01-Pouvoir-Coalitionnel-SMT-Python.ipynb), dont nous reprenons le noyau en place.*

Un président détient le plus gros poids d'une assemblée. La question contrefactuelle de Coalitions-01 : si son parti obtient **un vote de quota de plus** à faire adopter, que devient son pouvoir réel ? L'indice de Banzhaf compte pour chaque joueur les coalitions où son adhésion fait basculer la victoire — les *swings* — et normalise. C'est la garantie de niveau 1 : une énumération exacte, vérifiable par le SMT de 01 en seconde voie.

In [2]:
# Noyau repris de Coalitions-01 (VotingGame), compacte pour le capstone.
@dataclass(frozen=True)
class VotingGame:
    weights: tuple[int, ...]
    quota: int
    names: tuple[str, ...]

    @property
    def players(self):
        return range(len(self.weights))

    def weight(self, coalition):
        return sum(self.weights[i] for i in coalition)

    def wins(self, coalition):
        return self.weight(coalition) >= self.quota

    def coalitions(self):
        for size in range(len(self.weights) + 1):
            for coalition in combinations(self.players, size):
                yield frozenset(coalition)

    def minimal_winning(self):
        return [s for s in self.coalitions() if self.wins(s) and all(not self.wins(s - {i}) for i in s)]

    def is_dummy(self, i):
        return all(self.wins(s | {i}) == self.wins(s) for s in self.coalitions() if i not in s)


def swing_counts(game):
    counts = {i: 0 for i in game.players}
    for i in game.players:
        for s in game.coalitions():
            if i not in s and not game.wins(s) and game.wins(s | {i}):
                counts[i] += 1
    return counts


def banzhaf(game):
    """Banzhaf normalise : swings du joueur sur total des swings (cf. Coalitions-01, section 2)."""
    swings = swing_counts(game)
    total = sum(swings.values())
    return {game.names[i]: swings[i] / total for i in game.players}


president = VotingGame((3, 2, 1), 4, ("President", "B", "C"))
print("Jeu de depart [4; 3, 2, 1] — coalitions gagnantes minimales :")
print([[president.names[i] for i in s] for s in president.minimal_winning()])

Jeu de depart [4; 3, 2, 1] — coalitions gagnantes minimales :
[['President', 'B'], ['President', 'C']]


In [3]:
# L'experience : un seul vote de quota de plus (4 -> 5), memes poids.
avant = VotingGame((3, 2, 1), 4, ("President", "B", "C"))
apres = VotingGame((3, 2, 1), 5, ("President", "B", "C"))

rows = []
for label, jeu in (("quota 4", avant), ("quota 5", apres)):
    b = banzhaf(jeu)
    rows.append({
        "jeu": f"[{jeu.quota}; {', '.join(map(str, jeu.weights))}] ({label})",
        **{f"poids {nom}": w for nom, w in zip(jeu.names, jeu.weights)},
        **{f"banzhaf {nom}": round(b[nom], 3) for nom in jeu.names},
        "C nul": jeu.is_dummy(2),
    })
mesure = pd.DataFrame(rows).set_index("jeu").T
print(mesure.to_string())

jeu               [4; 3, 2, 1] (quota 4) [5; 3, 2, 1] (quota 5)
poids President                        3                      3
poids B                                2                      2
poids C                                1                      1
banzhaf President                    0.6                    0.5
banzhaf B                            0.2                    0.5
banzhaf C                            0.2                    0.0
C nul                              False                   True


### Lecture du résultat — un vote de quota, trois destins

La mesure ci-dessus est celle de Coalitions-01, resserrée sur un seul levier. Au quota 4, le président (poids 3, soit 50 % de l'assemblée) domine : son Banzhaf vaut 0,6 contre 0,2 pour chacun des autres. Au quota 5, **un seul vote de plus à réunir** :

- le président tombe de 0,6 à **0,5** : sa moitié de poids ne lui achète plus que la moitié du pouvoir ;
- B monte de 0,2 à **0,5** : à poids inférieur, pouvoir égal — les coalitions gagnantes passent toutes par l'un *ou* l'autre ;
- C tombe à **0,0** et devient nul (`is_dummy` vrai) : son poids existe, son pouvoir a disparu.

C'est le cœur de « poids n'est pas pouvoir » : la valeur d'un membre d'une coalition ne se lit pas dans son poids mais dans ses **swings** — les situations où son adhésion bascule la coalition. Coalitions-01 pousse cette énumération jusqu'à la garantie SMT (`UNSAT`/`SAT` sur les jeux croisés) ; le capstone s'arrête à l'indice exact, et renvoie au carnet source pour la seconde voie.

## 4. Expérience 2 — Cycle contre chaîne

*Ancrée dans [Coalitions-03](Coalitions/Coalitions-03-Echange-de-Reins-Lean-Python.ipynb), dont nous reprenons la construction du graphe.*

L'échange de reins est l'institution réelle de la sous-série : des paires donneur-receveur incompatibles entre elles cherchent un échange. Deux mécanismes s'affrontent : les **cycles** entre paires (chacune donne à la suivante, la boucle se referme) et les **chaînes** amorcées par un donneur altruiste (la chaîne peut s'arrêter sans refermer de boucle — un donneur « terminal » peut donner à la liste d'attente). La question de 03 : que change un altruiste, en cardinalité **et** en équité ? C'est la garantie de niveau 2 : une institution avec les contraintes du monde, où la couverture se mesure, ne se démontre pas.

In [4]:
# Noyau repris de Coalitions-03 : graphe de compatibilite, cycles, chaines altruistes.
def construire_graphe_avec_altruistes(paires, altruistes):
    g = nx.DiGraph()
    for i, (d, r) in enumerate(paires):
        g.add_node(("P", i), donneur=d, receveur=r, role="paire")
    for j, a in enumerate(altruistes):
        g.add_node(("A", j), donneur=a, receveur=None, role="altruiste")
    for j, a in enumerate(altruistes):
        for i, (d, r) in enumerate(paires):
            if a == r:
                g.add_edge(("A", j), ("P", i))
    for i, (d1, r1) in enumerate(paires):
        for k, (d2, r2) in enumerate(paires):
            if i != k and d1 == r2:
                g.add_edge(("P", i), ("P", k))
    return g


def couverture_par_cycles(g, k_max=3):
    """Couverture gloutonne : cycles de paires disjoints, les plus longs d'abord."""
    sous = g.subgraph([n for n, d in g.nodes(data=True) if d.get("role") == "paire"])
    candidats = sorted((tuple(c) for c in nx.simple_cycles(sous) if len(c) <= k_max), key=len, reverse=True)
    pris, pris_dans = [], set()
    for c in candidats:
        if not (set(c) & pris_dans):
            pris.append(c)
            pris_dans |= set(c)
    return pris


def enumerer_chaines(g, k_max=4):
    """Chaines demarrees par un altruiste (DFS bornee), reprises de 03 tel quel."""
    chaines = []
    for source in [n for n, d in g.nodes(data=True) if d.get("role") == "altruiste"]:
        def dfs(noeud, chemin, profondeur):
            if profondeur >= k_max:
                chaines.append(tuple(chemin))
                return
            for succ in g.successors(noeud):
                if succ not in chemin:
                    dfs(succ, chemin + [succ], profondeur + 1)
            if profondeur >= 1:
                chaines.append(tuple(chemin))
        dfs(source, [source], 0)
    return chaines


# Instance fixee (semence de 03, reproductible).
import random
rng = random.Random(12240)
groupes = ["O", "A", "B", "AB"]
paires = [(rng.choice(groupes), rng.choice(groupes)) for _ in range(6)]
altruistes = ["O", "O"]

g_sans = construire_graphe_avec_altruistes(paires, [])
g_avec = construire_graphe_avec_altruistes(paires, altruistes)
print("Paires (donneur, receveur) :", paires)
print("Altruistes :", altruistes)
print("Aretes sans altruiste :", g_sans.number_of_edges(), "| avec :", g_avec.number_of_edges())

Paires (donneur, receveur) : [('B', 'B'), ('AB', 'O'), ('B', 'B'), ('O', 'AB'), ('A', 'AB'), ('AB', 'B')]
Altruistes : ['O', 'O']
Aretes sans altruiste : 9 | avec : 11


In [5]:
# La mesure : cycles seuls, puis la chaine qui sert le PLUS DE PAIRES ORPHELINES
# (le plan realiste est sequentiel : cycles d'abord, chaine sur ce qui reste).
cycles_seuls = couverture_par_cycles(g_avec, k_max=3)
orphelines = {("P", i) for i in range(len(paires))} - {x for c in cycles_seuls for x in c}

chaines = enumerer_chaines(g_avec, k_max=4)
# parmi les chaines, celle qui couvre le plus d'orphelines (puis la plus longue)
def orphelines_de(ch):
    return {x for x in ch if x in orphelines}
meilleure = max(chaines, key=lambda ch: (len(orphelines_de(ch)), len(ch))) if chaines else ()
servies_avec_chaine = {x for c in cycles_seuls for x in c} | {x for x in meilleure if x[0] == "P"}

print("Couverture par cycles seuls :", [list(c) for c in cycles_seuls])
print("  paires servies :", len(orphelines) != len(paires) and len(paires) - len(orphelines), "/", len(paires))
chaines_utiles = [ch for ch in chaines if orphelines_de(ch)]
print("Chaines servant au moins une orpheline :", [list(ch) for ch in sorted(chaines_utiles, key=len, reverse=True)[:3]])
print("Chaine retenue :", list(meilleure), "-> orphelines servies :", sorted(orphelines_de(meilleure)))
print("Total avec chaine :", len(servies_avec_chaine), "/", len(paires),
      "| taux :", round(len(servies_avec_chaine) / len(paires), 3))
print("Paires toujours orphelines :", [paires[i] for i in range(len(paires)) if ("P", i) not in servies_avec_chaine])

Couverture par cycles seuls : [[('P', 2), ('P', 0)], [('P', 1), ('P', 3)]]
  paires servies : 4 / 6
Chaines servant au moins une orpheline : [[('A', 0), ('P', 1), ('P', 4)], [('A', 1), ('P', 1), ('P', 4)]]
Chaine retenue : [('A', 0), ('P', 1), ('P', 4)] -> orphelines servies : [('P', 4)]
Total avec chaine : 5 / 6 | taux : 0.833
Paires toujours orphelines : [('AB', 'B')]


### Lecture du résultat — ce que la chaîne déverrouille

La couverture par cycles seuls plafonne à **4/6** : deux boucles de deux paires (P0↔P2 et P1↔P3), et aucune boucle ne peut accueillir P4 — son receveur AB ne matche que les donneurs de P1 et P3, déjà engagés dans leur cycle. La chaîne retenue `[A0 → P1 → P4]` traverse une paire déjà servie (P1) et **termine sur l'orpheline P4** sans refermer de boucle : le donneur de la dernière paire peut donner à la liste d'attente. C'est précisément cette liberté de ne pas se refermer qui fait monter la couverture à **5/6** — une contrainte topologique levée, pas un simple gain en cardinalité.

La paire P5, elle, reste orpheline même avec chaîne : son receveur B ne matche (à groupe exact) que les donneurs de P0 et P2, déjà pris dans les cycles, et les altruistes disponibles — de groupe O — ne pointent ici que vers P1. Le prix de la chaîne se lit dans cette asymétrie : un donneur altruiste est rare, et son groupe décide de qui il débloque. La question d'arbitrage cardinalité/équité — quelles paires sacrifie-t-on quand la pool est trop petite — est celle de Coalitions-03, tranchée là-bas par le front de Pareto.

## 5. Expérience 3 — L'heuristique, l'optimum, le certificat

*Ancrée dans [Coalitions-04](Coalitions/Coalitions-04-Munkres-Affectation-Python.ipynb) (dont l'algorithme est repris intégralement) et [Coalitions-05](Coalitions/Coalitions-05-Lean-Affectation-Lean.ipynb) (dont le lake fournit le certificat).*

Trois réponses à « quelle est la meilleure affectation de n agents sur n tâches » : l'**heuristique** gloutonne (rapide, sans garantie), l'**optimum** de Kuhn-Munkres (garanti par le couple dual), et le **théorème** du lake (la garantie elle-même devient un objet machine-vérifié). Nous mettons les trois sur le **même objet** : une matrice 3×3, puis une série de matrices aléatoires seedées.

In [6]:
# Kuhn-Munkres repris INTEGRALEMENT de Coalitions-04 (methode hongroise pedagogique,
# arithmetique entiere exacte, chaque iteration construit une fois l'arbre hongrois).
def kuhn_munkres(C):
    """Retourne (valeur, assignment, u, v) avec u_i + v_j <= c_ij pour tout (i, j)."""
    C = np.asarray(C, dtype=np.int64)
    n = len(C)
    u = C.min(axis=1).astype(np.int64).copy()
    v = np.zeros(n, dtype=np.int64)
    match_col = [-1] * n
    match_row = [-1] * n

    def arbre_hongrois(i0):
        dec_j = {}
        lignes, colonnes = {i0}, set()
        file = [i0]
        while file:
            i = file.pop(0)
            for j in range(n):
                if j in colonnes or u[i] + v[j] != C[i, j]:
                    continue
                dec_j[j] = i
                colonnes.add(j)
                if match_col[j] == -1:
                    return dec_j, lignes, colonnes, j
                lignes.add(match_col[j])
                file.append(match_col[j])
        return dec_j, lignes, colonnes, None

    for i0 in range(n):
        while match_row[i0] == -1:
            dec_j, lignes, colonnes, j_libre = arbre_hongrois(i0)
            if j_libre is not None:
                j = j_libre
                i = dec_j[j]
                while True:
                    j_suiv = match_row[i]
                    match_col[j] = i
                    match_row[i] = j
                    if i == i0:
                        break
                    j = j_suiv
                    i = dec_j[j]
            else:
                hors = [j for j in range(n) if j not in colonnes]
                delta = min(int(C[i, j] - u[i] - v[j]) for i in lignes for j in hors)
                for i in lignes:
                    u[i] += delta
                for j in colonnes:
                    v[j] -= delta
    assert sorted(match_row) == list(range(n)), "pas un matching parfait"
    valeur = int(sum(int(C[i, match_row[i]]) for i in range(n)))
    return valeur, match_row, u, v


def glouton(C):
    """Glouton du CAPSTONE (a comparer a l'exercice 2 de 04) : chaque ligne prend
    sa colonne libre la moins chere, dans l'ordre."""
    C = np.asarray(C, dtype=np.int64)
    n = len(C)
    libres = list(range(n))
    assignment = []
    for i in range(n):
        j = min(libres, key=lambda k: C[i, k])
        libres.remove(j)
        assignment.append(j)
    return assignment, int(sum(int(C[i, assignment[i]]) for i in range(n)))


C3 = np.array([[4, 1, 3], [2, 0, 5], [3, 2, 2]])
val_km, aff_km, u3, v3 = kuhn_munkres(C3)
_, val_gl = glouton(C3)
print("Matrice C3 :", C3.tolist())
print("Glouton  : affectation", [int(j) + 1 for j in glouton(C3)[0]], "-> valeur", val_gl)
print("Munkres  : affectation", [int(j) + 1 for j in aff_km], "-> valeur", val_km)
print("Ecart glouton - optimum :", val_gl - val_km)

Matrice C3 : [[4, 1, 3], [2, 0, 5], [3, 2, 2]]
Glouton  : affectation [2, 1, 3] -> valeur 5
Munkres  : affectation [2, 1, 3] -> valeur 5
Ecart glouton - optimum : 0


In [7]:
# L'ecart systematique : 500 matrices aleatoires seedees, 4x4.
rng = np.random.default_rng(2026)
ecarts = []
for _ in range(500):
    M = rng.integers(0, 20, size=(4, 4))
    _, v_gl = glouton(M)
    v_km, _, _, _ = kuhn_munkres(M)
    ecarts.append(v_gl - v_km)
ecarts = np.array(ecarts)
print("500 matrices 4x4 (seed 2026) :")
print("  ecart moyen glouton - optimum :", round(float(ecarts.mean()), 3))
print("  ecart max :", int(ecarts.max()), "| matrices ou le glouton est optimal :", int((ecarts == 0).sum()), "/ 500")

500 matrices 4x4 (seed 2026) :
  ecart moyen glouton - optimum : 4.83
  ecart max : 30 | matrices ou le glouton est optimal : 186 / 500


In [8]:
# Le CERTIFICAT : pourquoi croire Munkres ? Le couple dual rendu par l'algorithme.
# Realisabilite duale : u_i + v_j <= c_ij partout ; valeur duale = somme(u) + somme(v).
# Si valeur duale = valeur primaire, l'ecart de dualite est nul : optimalite, le
# schema EXACT du theoreme Assignment.optimality_of_zero_gap du lake (Coalitions-05).
dual_ok = all(u3[i] + v3[j] <= C3[i, j] for i in range(3) for j in range(3))
dual_val = int(u3.sum() + v3.sum())
print("Potentiels duaux u :", u3.tolist(), "| v :", v3.tolist())
print("Realisabilite duale (u_i + v_j <= c_ij partout) :", dual_ok)
print("Valeur duale somme(u)+somme(v) :", dual_val, "| valeur primaire Munkres :", val_km)
print("Ecart de dualite :", val_km - dual_val, "-> optimalite certifiee" if val_km == dual_val else "-> ANOMALIE")

# Ce que le lake prouve, lu depuis ses sources (jamais execute ici — cf. 05) :
racine_lake = REPO_ROOT / "MyIA.AI.Notebooks/GameTheory/assignment_lean/Assignment/KuhnMunkres.lean"
if racine_lake.exists():
    source = racine_lake.read_text(encoding="utf-8")
    exposes = [l.split()[1] for l in source.splitlines()
               if l.startswith(("theorem ", "lemma ", "def "))]
    print("API exposee par KuhnMunkres.lean :", exposes)

Potentiels duaux u : [3, 2, 2] | v : [0, -2, 0]
Realisabilite duale (u_i + v_j <= c_ij partout) : True
Valeur duale somme(u)+somme(v) : 5 | valeur primaire Munkres : 5
Ecart de dualite : 0 -> optimalite certifiee
API exposee par KuhnMunkres.lean : ['EqEdge', 'kuhn_munkres_correct', 'dualFeasible_tighten']


### Lecture du résultat — trois garanties sur un seul objet

Sur C3, le glouton tombe sur l'optimum par chance (écart 0) — c'est possible, jamais garanti : la loterie se mesure sur les 500 matrices seedées, où le glouton n'est optimal que **186 fois sur 500**, avec un écart moyen de **4,83** et un pire cas à **30**. Munkres, lui, **rend la preuve avec le résultat** : le couple `(u, v)` est réalisable (`u_i + v_j <= c_ij`, vérifié ci-dessus en une ligne), et sa somme égale la valeur de l'affectation — écart de dualité nul, donc optimalité, par le théorème de dualité faible que le lake `assignment_lean` énonce (`Assignment.weak_duality`) et dont `Assignment.optimality_of_zero_gap` fait le critère. C'est le saut de garantie : le glouton *affirme*, Munkres *certifie*, le lake *démontre le certificat*. L'inventaire complet du lake fait l'objet de la section suivante.

## 6. Ce que le lake Lean prouve — inventaire mesuré

Le lake `assignment_lean` vit sous `MyIA.AI.Notebooks/GameTheory/assignment_lean/`. Nous l'inventorions **par lecture directe de ses sources** (le pattern de Coalitions-03 et 05) : modules, théorèmes, définitions, `sorry`. Le lake n'est jamais invoqué ici — sa preuve de build vit dans la CI de Coalitions-05.

In [9]:
# Inventaire par lecture directe des sources (pattern Coalitions-03/05, sans lake).
import re

lake = REPO_ROOT / "MyIA.AI.Notebooks/GameTheory/assignment_lean/Assignment"
lignes_inventaire = []
total_thm, total_def, total_sorry = 0, 0, 0
for f in sorted(lake.glob("*.lean")):
    if f.stem.endswith("_en"):
        continue  # jumeaux de traduction — comptes a part
    src = f.read_text(encoding="utf-8")
    # commentaire-strip AVANT de compter sorry (cf. regle anti-regression : jamais grep -c sorry)
    sans_commentaires = re.sub(r"/-.*?-/", "", src, flags=re.S)
    sans_commentaires = re.sub(r"--.*$", "", sans_commentaires, flags=re.M)
    n_thm = len(re.findall(r"^(?:theorem|lemma)\s+\w+", src, flags=re.M))
    n_def = len(re.findall(r"^def\s+\w+", src, flags=re.M))
    n_sorry = len(re.findall(r"^\s*sorry\b", sans_commentaires, flags=re.M))
    total_thm += n_thm; total_def += n_def; total_sorry += n_sorry
    lignes_inventaire.append({"module": f.stem, "theoremes": n_thm, "defs": n_def, "sorry": n_sorry})

jumeaux_en = len(list(lake.glob("*_en.lean")))
print(pd.DataFrame(lignes_inventaire).to_string(index=False))
print(f"Totaux FR : {total_thm} theoremes, {total_def} definitions, {total_sorry} sorry | jumeaux _en : {jumeaux_en}")

     module  theoremes  defs  sorry
   Counting          4     2      0
Definitions          0     2      0
    Duality          1     2      0
KuhnMunkres          2     1      0
 Optimality          2     0      0
Totaux FR : 9 theoremes, 7 definitions, 0 sorry | jumeaux _en : 5


### Lecture de l'inventaire — la colonne vertébrale de la garantie

L'inventaire mesuré ci-dessus donne la structure de la preuve : les **définitions** posent les objets (`Assignment.value`, la valeur d'une affectation ; `DualFeasible`, la réalisabilité duale), les **théorèmes** enchaînent les garanties — `weak_duality` (toute affectation vaut au plus tout dual réalisable), `optimality_of_zero_gap` (écart nul = optimalité : le critère invoqué à l'expérience 3), `kuhn_munkres_correct` (l'algorithme lui-même, prouvé) — et la colonne `sorry` à **zéro** dit qu'aucune maille n'est laissée en trémie : ce qui est énoncé est démontré. Chaque module FR a son jumeau `_en` byte-identique hors commentaires (la convention i18n du lac). C'est le niveau de garantie terminal : pas « l'algorithme est correct » affirmé, mais **démontré par un noyau qui ne croit que ce qu'il vérifie** — et ce noyau, c'est la CI de [Coalitions-05](Coalitions/Coalitions-05-Lean-Affectation-Lean.ipynb) qui le relit à chaque changement.

## 7. Par où y entrer

Trois parcours dans la sous-série, selon le temps et la question :

- **Une heure, le pouvoir** : [Coalitions-01](Coalitions/Coalitions-01-Pouvoir-Coalitionnel-SMT-Python.ipynb) seul — les trois indices, la seconde voie SMT, la cross-validation sur neuf couples. Il referme la question « poids contre pouvoir » ouverte par l'expérience 1.
- **Deux heures, l'institution** : [01](Coalitions/Coalitions-01-Pouvoir-Coalitionnel-SMT-Python.ipynb) puis [03](Coalitions/Coalitions-03-Echange-de-Reins-Lean-Python.ipynb) — du pouvoir mesuré à l'arbitrage cardinalité/équité d'un programme d'échanges réel, dont l'expérience 2 a montré le levier altruiste.
- **La boucle entière** : [01](Coalitions/Coalitions-01-Pouvoir-Coalitionnel-SMT-Python.ipynb) → [04](Coalitions/Coalitions-04-Munkres-Affectation-Python.ipynb) → [05](Coalitions/Coalitions-05-Lean-Affectation-Lean.ipynb) — l'affectation depuis l'heuristique jusqu'au théorème machine-vérifié, bouclée par [02](Coalitions/Coalitions-02-Assistance-Games-Python.ipynb) qui replace la coopération dans le temps (bornes no-regret).

## 8. Exercices

Trois exercices non résolus, un par palier de garantie. Les stubs s'exécutent sans erreur ; à vous de les remplir.

In [10]:
# Exercice 1 (suite directe de 01, exercice 2) : le jeu de rupture.
# Un jeu ou un joueur de poids MINIMAL detient un pouvoir de veto mesurable.
# TODO etudiant :
# Etape 1 : construire un VotingGame de la forme [q; 1, 1, 1, 1, w] ou le petit
#           poids w fait de son porteur un joueur veto (indice : les quatre autres
#           ensemble ne font quota que si q > 4).
# Etape 2 : mesurer banzhaf(jeu) et verifier que is_veto... — le capstone n'a garde
#           que is_dummy ; ajoutez le predicat is_veto du carnet 01.
# Etape 3 : interpreter : quel est le Banzhaf d'un veto, et pourquoi ?

def jeu_de_rupture():
    # TODO etudiant : retourner le VotingGame et le dictionnaire banzhaf mesure.
    print("Exercice 1 a completer : jeu_de_rupture()")

jeu_de_rupture()

Exercice 1 a completer : jeu_de_rupture()


In [11]:
# Exercice 2 (a partir de 02) : le budget de l'apprenant.
# EXP3 garantit un regret en racine(T) ; que reste-t-il avec un budget d'essais borne ?
# TODO etudiant :
# Etape 1 : relire la boucle EXP3 de Coalitions-02 (section bench).
# Etape 2 : la borne a T/2 essais effectifs et mesurer le regret empirique moyen
#           sur 200 repetitions (meme oracle hindsight que 02).
# Etape 3 : comparer a la borne theorique en racine(T) : la perte est-elle en
#           constante multiplicative ou en ordre de grandeur ?

def regret_budget():
    # TODO etudiant : retourner (regret_complet, regret_budget).
    print("Exercice 2 a completer : regret_budget()")

regret_budget()

Exercice 2 a completer : regret_budget()


In [12]:
# Exercice 3 (04/05) : l'instance ou le glouton egale Munkres — et pourquoi le certificat reste du.
# TODO etudiant :
# Etape 1 : reprendre la boucle de l'experience 3 et isoler les matrices 3x3 OU le
#           glouton est optimal (ecart nul) — il en existe (mesure ci-dessus).
# Etape 2 : pour l'une d'elles, executer kuhn_munkres et verifier le certificat
#           dual (realisabilite + ecart nul), comme dans la cellule du certificat.
# Etape 3 : expliquer en une phrase : si le glouton est optimal sur CETTE matrice,
#           qu'est-ce que le certificat ajoute quand meme que la valeur egaleie ne prouve pas ?
#           (indice : une egalite mesuree sur une instance n'est pas une garantie sur toutes.)

def glouton_optimal_instance():
    # TODO etudiant : retourner (M, valeur_glouton, valeur_munkres, certificat_ok).
    print("Exercice 3 a completer : glouton_optimal_instance()")

glouton_optimal_instance()

Exercice 3 a completer : glouton_optimal_instance()


## À retenir

La sous-série Coalitions répond à « combien vaut une coalition » par une **échelle de garanties**, et le capstone a fait monter trois de ses barreaux sur les mêmes objets :

1. **Mesuré** — le pouvoir contrefactuel (Banzhaf) : un vote de quota de plus et le président à 60 % du poids tombe à égalité de pouvoir, le troisième devient nul. La valeur d'un membre est dans ses swings, pas dans son poids ([01](Coalitions/Coalitions-01-Pouvoir-Coalitionnel-SMT-Python.ipynb)).
2. **Institutionnel** — la chaîne altruiste déverrouille ce que le cycle ne peut pas refermer : la couverture monte, l'arbitrage cardinalité/équité reste ([03](Coalitions/Coalitions-03-Echange-de-Reins-Lean-Python.ipynb)).
3. **Certifié, démontré** — le glouton mesure, Munkres certifie (`u_i + v_j <= c_ij` + écart nul), le lake démontre le critère du certificat, sans un `sorry` ([04](Coalitions/Coalitions-04-Munkres-Affectation-Python.ipynb), [05](Coalitions/Coalitions-05-Lean-Affectation-Lean.ipynb)).

Et entre les deux, [02](Coalitions/Coalitions-02-Assistance-Games-Python.ipynb) rappelle que la coopération a aussi un prix temporel : la borne no-regret est le niveau de garantie de l'apprenant qui n'a pas d'oracle. Cinq carnets, une échelle : chaque fois que la question « est-ce prouvé ? » se pose, la sous-série a un barreau à offrir.